# Data Preprocessing for Loan Risk Prediction

This notebook performs:
- dataset loading
- data cleaning
- missing value handling
- target variable creation
- preprocessing preparation

In [ ]:
import pandas as pd
import numpy as np

## Load Dataset

In [ ]:


df = pd.read_csv(
    r"C:\Users\chand\OneDrive\Desktop\lending club dataset\sampled_100k_rows.csv",
    encoding="utf-8"
)


print("Initial shape:", df.shape)


Initial shape: (100000, 151)


C:\Users\chand\AppData\Local\Temp\ipykernel_4736\455521396.py:1: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


## Dataset Overview

In [55]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,60577368,NaN,15000,15000,15000,60 months,11.53,330.12,B,B5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,59112798,NaN,25600,25600,25600,60 months,27.88,795.23,G,G3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,65302450,NaN,6000,6000,6000,36 months,15.41,209.20,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,65474680,NaN,5000,5000,5000,36 months,9.99,161.32,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,66430368,NaN,12000,12000,12000,60 months,8.38,245.51,B,B1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


## Remove Identification Features

In [ ]:
identification_cols = [
    "id",
    "member_id",
    "url",
    "title"
]

df.drop(columns=identification_cols, inplace=True, errors="ignore")

print("After dropping identification columns:", df.shape)


After dropping identification columns: (100000, 147)


## Remove Irrelevant Features

In [ ]:
payment_timeline_cols = [
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_interest",
    "total_rec_late_fee"
]

df.drop(columns=payment_timeline_cols, inplace=True, errors="ignore")

print("After dropping payment/timeline columns:", df.shape)


After dropping payment/timeline columns: (100000, 140)


In [ ]:
settlement_recovery_cols = [
    "recoveries",
    "collection_recovery_fee",
    "debt_settlement_flag",
    "debt_settlement_flag_date",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term"
]

df.drop(columns=settlement_recovery_cols, inplace=True, errors="ignore")

print("After dropping settlement/recovery columns:", df.shape)


After dropping settlement/recovery columns: (100000, 131)


In [ ]:
hardship_cols = [col for col in df.columns if col.startswith("hardship")]

df.drop(columns=hardship_cols, inplace=True, errors="ignore")

print("After dropping hardship columns:", df.shape)


After dropping hardship columns: (100000, 119)


## Create Target Variable

In [ ]:
df["loan_default"] = df["loan_status"].apply(
    lambda x: 0 if x == "Fully Paid" else 1
)

df.drop(columns=["loan_status"], inplace=True)

print("Target created. Shape:", df.shape)


Target created. Shape: (100000, 119)


## Handle Missing Values

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])


In [ ]:
 # Drop columns with more than 70% missing values
missing_threshold = 0.7

missing_ratio = df.isna().mean()
high_missing_cols = missing_ratio[missing_ratio > missing_threshold].index.tolist()

print("Dropping columns:", high_missing_cols)

df.drop(columns=high_missing_cols, inplace=True)


Dropping columns: ['revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog']


In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)


loan_amnt                   0
num_accts_ever_120_pd       0
mths_since_recent_inq       0
mths_since_recent_bc_dlq    0
mths_since_recent_bc        0
mort_acc                    0
mo_sin_rcnt_tl              0
mo_sin_rcnt_rev_tl_op       0
mo_sin_old_rev_tl_op        0
mo_sin_old_il_acct          0
dtype: int64

In [ ]:
df.to_csv("loan_default_final_ready.csv", index=False)
print("Final corrected CSV saved")


✅ Final corrected CSV saved
